# Exercice 3 — Mini-BERT et mots masqués

Ce notebook entraîne un petit Transformer bidirectionnel sur un corpus de phrases. Il sélectionne 15 % des mots ordinaires, applique la règle 80/10/10 de BERT, puis calcule la perte uniquement sur les positions sélectionnées.

In [1]:
import random
import re
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAD, UNK, CLS, SEP, MASK = 0, 1, 2, 3, 4
specials = {"[PAD]": PAD, "[UNK]": UNK, "[CLS]": CLS, "[SEP]": SEP, "[MASK]": MASK}
max_length = 32

phrases = [
    "le chat dort sur le canapé", "le chien joue dans le jardin",
    "la lune éclaire la nuit", "le soleil chauffe la terre",
    "les enfants lisent un livre", "une voiture traverse la ville",
    "la musique accompagne le travail", "un oiseau chante dans l arbre",
    "le modèle comprend le contexte", "le transformeur utilise l attention",
    "les données servent à apprendre", "un réseau transforme les tokens",
    "la classe prépare un exercice", "le professeur explique le modèle",
    "un livre décrit une histoire", "la machine analyse une phrase",
]
train_phrases, val_phrases = phrases[:12], phrases[12:]

def mots(phrase):
    return re.findall(r"[A-Za-zÀ-ÿ']+", phrase.lower())

vocabulaire = dict(specials)
for phrase in train_phrases:
    for mot in mots(phrase):
        if mot not in vocabulaire:
            vocabulaire[mot] = len(vocabulaire)
inv_vocabulaire = {i: mot for mot, i in vocabulaire.items()}
taille_vocabulaire = len(vocabulaire)

def encode(phrase):
    ids = [CLS] + [vocabulaire.get(mot, UNK) for mot in mots(phrase)] + [SEP]
    ids += [PAD] * (max_length - len(ids))
    return ids[:max_length]

train_ids = torch.tensor([encode(p) for p in train_phrases], dtype=torch.long)
val_ids = torch.tensor([encode(p) for p in val_phrases], dtype=torch.long)
print("Vocabulaire :", taille_vocabulaire, "| formes :", train_ids.shape, val_ids.shape)

Vocabulaire : 50 | formes : torch.Size([12, 32]) torch.Size([4, 32])


## 1. Corrompre les phrases selon la règle 80/10/10

Les labels conservent les tokens originaux. Les positions non sélectionnées reçoivent `-100` afin d'être ignorées par `CrossEntropyLoss`.

In [2]:
def corruption(input_ids):
    input_ids = input_ids.clone()
    labels = torch.full_like(input_ids, -100)
    selection = torch.zeros_like(input_ids, dtype=torch.bool)
    for ligne in range(input_ids.shape[0]):
        positions = [
            i for i, token in enumerate(input_ids[ligne].tolist())
            if token not in (PAD, CLS, SEP, MASK)
        ]
        nombre = max(1, round(0.15 * len(positions)))
        choisies = random.sample(positions, min(nombre, len(positions)))
        selection[ligne, choisies] = True
        labels[ligne, choisies] = input_ids[ligne, choisies]
        for position in choisies:
            tirage = random.random()
            if tirage < 0.8:
                input_ids[ligne, position] = MASK
            elif tirage < 0.9:
                candidats = list(range(5, taille_vocabulaire))
                input_ids[ligne, position] = random.choice(candidats)
    return input_ids, labels, selection

corrompus, labels, selection = corruption(train_ids[:2])
print("Sélection par phrase :", selection.sum(dim=1).tolist())
print("Labels ignorés :", (labels == -100).sum().item())

Sélection par phrase : [1, 1]
Labels ignorés : 62


## 2. Modèle bidirectionnel

Il n'y a pas de masque causal : chaque token peut utiliser le contexte à gauche et à droite. `src_key_padding_mask` empêche l'attention d'utiliser les positions `[PAD]`.

In [3]:
class MiniBERT(nn.Module):
    def __init__(self, vocabulaire, longueur_max=32):
        super().__init__()
        self.tokens = nn.Embedding(vocabulaire, 64)
        self.positions = nn.Embedding(longueur_max, 64)
        couche = nn.TransformerEncoderLayer(
            d_model=64, nhead=4, dim_feedforward=256,
            dropout=0.1, batch_first=True,
        )
        self.encodeur = nn.TransformerEncoder(couche, num_layers=2)
        self.sortie = nn.Linear(64, vocabulaire)

    def forward(self, input_ids):
        _, longueur = input_ids.shape
        positions = torch.arange(longueur, device=input_ids.device)
        z = self.tokens(input_ids) + self.positions(positions)[None, :, :]
        masque_padding = input_ids.eq(PAD)
        z = self.encodeur(z, src_key_padding_mask=masque_padding)
        return self.sortie(z)


modele = MiniBERT(taille_vocabulaire).to(device)
print("Logits :", tuple(modele(train_ids[:2].to(device)).shape))

Logits : (2, 32, 50)


## 3. Entraîner sur les positions sélectionnées

Les données sont remasquées à chaque étape. Les positions conservées inchangées restent bien dans la perte, comme demandé.

In [4]:
def calcul_perte(ids):
    inputs, labels, selection = corruption(ids)
    logits = modele(inputs.to(device))
    perte = F.cross_entropy(
        logits.reshape(-1, taille_vocabulaire),
        labels.to(device).reshape(-1),
        ignore_index=-100,
    )
    return perte


optimiseur = torch.optim.AdamW(modele.parameters(), lr=2e-3)
for etape in range(300):
    modele.train()
    perte = calcul_perte(train_ids)
    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()
    if (etape + 1) % 50 == 0:
        print(f"Étape {etape + 1:03d} | perte : {perte.item():.4f}")

Étape 050 | perte : 1.5514
Étape 100 | perte : 0.3619
Étape 150 | perte : 0.1529
Étape 200 | perte : 0.0881
Étape 250 | perte : 0.0488
Étape 300 | perte : 0.0990


## 4. Exactitude sur la validation et propositions top-3

In [5]:
def evaluation(ids):
    modele.eval()
    total_correct, total = 0, 0
    with torch.no_grad():
        for _ in range(20):
            inputs, labels, selection = corruption(ids)
            logits = modele(inputs.to(device)).cpu()
            positions = selection
            predictions = logits.argmax(dim=-1)
            total_correct += (predictions[positions] == labels[positions]).sum().item()
            total += positions.sum().item()
    return total_correct / total


print(f"Exactitude validation : {evaluation(val_ids):.2%}")

Exactitude validation : 6.25%


c:\Users\Lenovo\Downloads\Architecture-Transformer\.venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


In [6]:
def affiche_top3(phrase):
    originale = encode(phrase)
    inputs = torch.tensor([originale], dtype=torch.long)
    positions = [i for i, token in enumerate(originale) if token not in (PAD, CLS, SEP)]
    if not positions:
        return
    position = positions[0]
    vrai_token = originale[position]
    inputs[0, position] = MASK
    modele.eval()
    with torch.no_grad():
        probabilites = F.softmax(modele(inputs.to(device))[0, position], dim=-1).cpu()
    valeurs, indices = torch.topk(probabilites, 3)
    print("Phrase :", phrase, "| mot masqué :", inv_vocabulaire[vrai_token])
    for probabilite, indice in zip(valeurs, indices):
        print(f"  {inv_vocabulaire[indice.item()]} : {probabilite.item():.2%}")


for phrase in val_phrases:
    affiche_top3(phrase)

Phrase : la classe prépare un exercice | mot masqué : la
  le : 63.84%
  les : 28.46%
  un : 1.52%
Phrase : le professeur explique le modèle | mot masqué : le
  le : 99.33%
  un : 0.19%
  travail : 0.05%
Phrase : un livre décrit une histoire | mot masqué : un
  le : 91.89%
  la : 2.16%
  un : 1.45%
Phrase : la machine analyse une phrase | mot masqué : la
  le : 92.62%
  la : 2.98%
  ville : 1.01%


## 5. Vérifier le masque de padding

Changer les valeurs situées dans le padding ne doit pas modifier les prédictions des tokens utiles.

In [8]:
modele.eval()
exemple = val_ids[:1].to(device)
modifie = exemple.clone()
positions_padding = exemple.eq(PAD)
modifie[positions_padding] = UNK
with torch.no_grad():
    logits_a = modele(exemple)
    logits_b = modele(modifie)
positions_utiles = ~positions_padding
difference = (logits_a[positions_utiles] - logits_b[positions_utiles]).abs().max().item()
print(f"Différence sur les tokens utiles : {difference:.8f}")
def forward(self, input_ids, masque_padding=None):
    if masque_padding is None:
        masque_padding = input_ids.eq(PAD)

    _, longueur = input_ids.shape
    positions = torch.arange(longueur, device=input_ids.device)
    z = self.tokens(input_ids) + self.positions(positions)[None, :, :]
    z = self.encodeur(z, src_key_padding_mask=masque_padding)
    return self.sortie(z)

Différence sur les tokens utiles : 3.36366153


## Questions — réponses

- **Pourquoi BERT utilise-t-il le contexte à droite ?**  
  Son attention est bidirectionnelle : elle n'impose pas de masque causal, donc un mot masqué peut exploiter les tokens précédents et suivants.

- **Pourquoi calculer la perte uniquement sur les positions sélectionnées ?**  
  L'objectif est de reconstruire les mots masqués. Inclure toutes les positions ferait surtout apprendre au modèle à recopier les tokens déjà visibles.

- **Pourquoi BERT ne génère-t-il pas comme GPT ?**  
  BERT est entraîné avec une tâche de reconstruction bidirectionnelle, pas avec la prédiction séquentielle du prochain token. Pour générer une séquence, GPT utilise au contraire une attention causale et produit les tokens un par un.